# Overpass extracts: shopping centres, local parks, medical centres

Queries OpenStreetMap once for `shop=mall`, `leisure=park`,
`amenity=clinic` and `amenity=doctors` within Victoria, then splits and
exports each as its own file.

## Imports and configuration

In [22]:
import numpy as np
import pandas as pd
import requests

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
USER_AGENT = "vic-overpass-extracts-nikhil_gaba"

VIC_LAT_RANGE = (-39.3, -33.9)
VIC_LON_RANGE = (140.9, 150.1)

SHOPPING_CENTRES_PATH = "../../external_data/shopping_centres.csv"
LOCAL_PARKS_PATH = "../../external_data/parks_local_park.csv"
MEDICAL_CENTRES_PATH = "../../external_data/medical_centres.csv"

OVERPASS_QUERY = """
[out:json][timeout:300];
area["ISO3166-2"="AU-VIC"]->.victoria;
(
  nwr["shop"="mall"](area.victoria);
  nwr["leisure"="park"](area.victoria);
  nwr["amenity"="clinic"](area.victoria);
  nwr["amenity"="doctors"](area.victoria);
);
out tags center;
"""

## Query Overpass once for every tag

A single request returns every element tagged with any of the four
tags. Each element's own tags are used afterwards to sort it into the
right output.

If the request fails or times out, the server may be busy; wait a
minute and rerun. If zero elements come back, the area lookup failed;
replace the `area["ISO3166-2"="AU-VIC"]` line with
`area["name"="Victoria"]["admin_level"="4"]->.victoria;`.

In [23]:
response = requests.post(
    OVERPASS_URL,
    data={"data": OVERPASS_QUERY},
    headers={"User-Agent": USER_AGENT},
    timeout=305,
)
response.raise_for_status()
elements = response.json()["elements"]

print(f"{len(elements)} elements returned")

16502 elements returned


## Flatten and split by tag

`element_to_row` is shared by every extract; the `id_prefix` argument
keeps each ID space distinct. Each element is routed to shopping
centres, local parks or medical centres based on its own tags. An
element matching none of the four tags (which shouldn't happen, given
the query) is counted and skipped rather than silently dropped.

In [24]:
def element_to_row(element, id_prefix):
    """Flatten one Overpass element into an ID, name and location.

    Nodes carry their coordinates directly. Ways and relations carry a
    computed centre point instead, under the key "center".
    """
    location = element.get("center", element)
    return {
        "Facility ID": f"{id_prefix}_{element['type']}_{element['id']}",
        "name": element.get("tags", {}).get("name"),
        "Latitude": location["lat"],
        "Longitude": location["lon"],
    }


shopping_centre_rows = []
local_park_rows = []
medical_centre_rows = []
n_unmatched = 0

for element in elements:
    tags = element.get("tags", {})
    if tags.get("shop") == "mall":
        shopping_centre_rows.append(element_to_row(element, "MALL"))
    elif tags.get("leisure") == "park":
        local_park_rows.append(element_to_row(element, "OSM_PARK"))
    elif tags.get("amenity") in ("clinic", "doctors"):
        medical_centre_rows.append(element_to_row(element, "MEDICAL"))
    else:
        n_unmatched += 1

shopping_centres = pd.DataFrame(shopping_centre_rows)
local_parks = pd.DataFrame(local_park_rows)
local_parks["PARK_CATEGORY"] = "local_park"
medical_centres = pd.DataFrame(medical_centre_rows)

print(f"Shopping centres: {len(shopping_centres)}")
print(f"Local parks: {len(local_parks)}")
print(f"Medical centres: {len(medical_centres)}")
print(f"Unmatched (unexpected): {n_unmatched}")

Shopping centres: 383
Local parks: 14667
Medical centres: 1452
Unmatched (unexpected): 0


## Check each extract

Same checks for every table: missing coordinates, unnamed entries,
duplicate names (an element mapped as both a node and a way shows up
twice), and rows outside Victoria's bounding box.

In [25]:
def check_and_clean(df, label):
    """Print basic quality checks and drop rows outside Victoria."""
    print(f"--- {label} ---")
    print(df[["Latitude", "Longitude"]].isna().sum())

    is_in_victoria = (
        df["Latitude"].between(*VIC_LAT_RANGE)
        & df["Longitude"].between(*VIC_LON_RANGE)
    )
    print(f"Outside Victoria: {(~is_in_victoria).sum()}")
    return df[is_in_victoria].reset_index(drop=True)


shopping_centres = check_and_clean(shopping_centres, "Shopping centres")
local_parks = check_and_clean(local_parks, "Local parks")
medical_centres = check_and_clean(medical_centres, "Medical centres")

--- Shopping centres ---
Latitude     0
Longitude    0
dtype: int64
Outside Victoria: 0
--- Local parks ---
Latitude     0
Longitude    0
dtype: int64
Outside Victoria: 0
--- Medical centres ---
Latitude     0
Longitude    0
dtype: int64
Outside Victoria: 0


### Deduplicate by location

The same place can be mapped twice in OSM (for example as both a node
and a building outline), which produces two rows with different
`Facility ID`s but coordinates a few metres apart. Rather than matching
on name, which risks merging genuinely different places that happen to
share a generic name, duplicates are found by proximity: any pair
within `DEDUP_DISTANCE_M` of each other is treated as the same place,
and only the first of each group is kept.

In [26]:
from sklearn.neighbors import BallTree

EARTH_RADIUS_M = 6_371_000
DEDUP_DISTANCE_M = 50


def deduplicate_by_location(df, label, distance_m=DEDUP_DISTANCE_M):
    """Drop rows whose coordinates fall within distance_m of an earlier
    row in the same table, treating them as the same real-world place.
    """
    if len(df) < 2:
        return df

    coordinates_rad = np.radians(df[["Latitude", "Longitude"]].to_numpy())
    tree = BallTree(coordinates_rad, metric="haversine")
    radius_rad = distance_m / EARTH_RADIUS_M

    is_duplicate = np.zeros(len(df), dtype=bool)
    for row in range(len(df)):
        if is_duplicate[row]:
            continue
        neighbours = tree.query_radius(
            coordinates_rad[[row]], r=radius_rad
        )[0]
        later_neighbours = neighbours[neighbours > row]
        is_duplicate[later_neighbours] = True

    n_dropped = is_duplicate.sum()
    print(f"{label}: dropping {n_dropped} of {len(df)} as duplicates")
    return df[~is_duplicate].reset_index(drop=True)


shopping_centres = deduplicate_by_location(
    shopping_centres, "Shopping centres"
)
local_parks = deduplicate_by_location(local_parks, "Local parks")
medical_centres = deduplicate_by_location(medical_centres, "Medical centres")

Shopping centres: dropping 26 of 383 as duplicates
Local parks: dropping 254 of 14667 as duplicates
Medical centres: dropping 196 of 1452 as duplicates


# Export all to CSV

In [27]:
shopping_centres.to_csv(SHOPPING_CENTRES_PATH, index=False)
local_parks.to_csv(LOCAL_PARKS_PATH, index=False)
medical_centres.to_csv(MEDICAL_CENTRES_PATH, index=False)